# DIMER video scrubber

Interactive frame-by-frame viewer **and** player for the synthetic video sets produced by the DLI stage.

**Use it as the visual spot-check for generated data:** scrub or play a few sims at each duration and confirm every frame shows particle dynamics. A truncated or empty frame (background noise only, no PSF spots) is immediately obvious here — the cheap inline guards should already prevent such data from being written, this is the human-eye backstop.

**How to run.** Activate the project env (`SRM_AND_SBI_ENVY_V0`), set `MACHINE_PROFILE` to the machine holding the data, and launch Jupyter from the repo root:

- **Locally (data on this machine):** `conda activate SRM_AND_SBI_ENVY_V0 && cd .../srm-and-sbi-dimer-alp && MACHINE_PROFILE=<your_profile> jupyter lab`
- **Remote (data on a server):** on the server run `MACHINE_PROFILE=<your_profile> jupyter lab --no-browser --port 8888`, then from your local machine tunnel `ssh -L 8888:localhost:8888 -p <ssh-port> <user>@<host>` and open the printed URL.

Set the **task, duration, and split** in the next cell to choose which video set to view.

In [ ]:
%matplotlib inline
import os

# The active machine profile selects the data-bank location. Set it here or in
# the shell before launching Jupyter.
os.environ.setdefault("MACHINE_PROFILE", "<your_profile>")

import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact

from srm_and_sbi_dimer_alp.parameterization import PARAMETERS
from srm_and_sbi_dimer_alp.visualization_dli import load_video_set

In [ ]:
# Choose which task's video set to view, and the RUN DURATION that produced it.
# The duration sets the timing label (e.g. 5.0 -> "5S_50FPS"). The global
# PARAMETERS.simulation.timing now holds only the fixed frame cadence, so the
# per-run duration is given explicitly via a RunTiming.
from srm_and_sbi_dimer_alp.parameterization import RunTiming

TASK = 0
TOTAL_TIME_SECONDS = 5.0   # <-- set to the duration of the videos to view (e.g. 1.0, 2.0, 5.0, 10.0)
SPLIT = "TRAIN"            # TRAIN / TEST / EVAL

timing = RunTiming(
    total_time_seconds=TOTAL_TIME_SECONDS, frames=PARAMETERS.simulation.timing,
)
timing_label = timing.label
data_bank_root = PARAMETERS.machine.data_bank_root
video_set_path = PARAMETERS.paths.video_set_path(
    TASK, data_bank_root, timing_label, compress=True, split=SPLIT,
)
print("Loading:", video_set_path)

videos = load_video_set(video_set_path)  # lazy handle; shape (n_sims, n_frames, H, W)
n_sims, n_frames = videos.shape[0], videos.shape[1]
print(f"{n_sims} sims x {n_frames} frames, frame {videos.shape[2]}x{videos.shape[3]}")

In [ ]:
# Drag the sliders to scrub through simulations and frames.
def show(sim, frame):
    plt.figure(figsize=(6, 6))
    plt.imshow(videos[sim, frame], cmap="magma", origin="lower",
               interpolation="none")
    plt.title(f"sim {sim}/{n_sims - 1}   frame {frame}/{n_frames - 1}")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.show()


interact(
    show,
    sim=IntSlider(min=0, max=n_sims - 1, step=1, value=0, description="sim"),
    frame=IntSlider(min=0, max=n_frames - 1, step=1, value=n_frames // 2,
                    description="frame"),
);

## Video playback

Pick a simulation and play it as a movie. Playback runs at the data's **native frame rate** (e.g. 50 fps for `*_50FPS` videos), so it plays in real time; the embedded player also has play / pause, loop, a speed control, and its own frame slider. Building the player pre-renders every frame, so a 10 s / 500-frame clip takes a few seconds to appear.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

PLAYBACK_SIM = 0   # which simulation to play

# Play back at the video's own acquisition rate (real time), derived from the
# simulation timing -- e.g. 50 for *_50FPS data. Do NOT hard-code this: it must
# match the frame rate the video was generated at.
FPS = round(1.0 / PARAMETERS.simulation.timing.frame_time_seconds)

clip = videos[PLAYBACK_SIM]   # load one sim's frames: (n_frames, H, W)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(clip[0], cmap="magma", origin="lower", interpolation="none",
               vmin=0, vmax=int(clip.max()))   # fixed scale: brightness comparable across frames
ax.set_title(f"sim {PLAYBACK_SIM}   frame 0/{n_frames - 1}   ({FPS} fps)")
plt.close(fig)   # suppress the static duplicate; the player below renders it


def _update(frame):
    im.set_data(clip[frame])
    ax.set_title(f"sim {PLAYBACK_SIM}   frame {frame}/{n_frames - 1}   ({FPS} fps)")
    return (im,)


anim = animation.FuncAnimation(
    fig, _update, frames=n_frames, interval=1000.0 / FPS, blit=False,
)
HTML(anim.to_jshtml())